In [ ]:
import os
import pandas as pd
import cv2 as cv
import matplotlib.pyplot as plt
from tqdm import tqdm

from segmentation import cloth_segmentation

PREPROCESSED_PATH = './data/preprocessed/'
os.makedirs(PREPROCESSED_PATH, exist_ok=True)
IMG_PATH = './data/images/'
os.makedirs(IMG_PATH, exist_ok=True)

In [ ]:
# df = pd.read_csv("./csvs_datasets/1000_per_color.csv")
df = pd.read_csv("./raw_data/cleaned_dataset.csv")
df = df[(~df["Color"].isna()) & (~df["Color"].isin(["Gold", "Silver"]))]
df.shape

In [ ]:
os.makedirs(PREPROCESSED_PATH, exist_ok=True)
for i, row in tqdm(df.iterrows(), total=df.shape[0]):
    file_name = row["file_name"]
    
    save_path = os.path.join(PREPROCESSED_PATH, file_name)
    if os.path.exists(save_path):
        continue
    
    image_path = os.path.join(IMG_PATH, file_name)
    if not os.path.exists(image_path):
        print(f"Image {image_path} not found.")
        continue
    
    image = cv.cvtColor(cv.imread(image_path), cv.COLOR_BGR2RGB)
    
    # Verifica se a imagem foi lida corretamente
    if image is None:
        print(f"Error reading image {image_path}.")
        continue
    
    segmented_image = cloth_segmentation(image, 0.5)
    
    if segmented_image is None:
        print(f"No segmentation for {file_name}.")
        continue
    
    cv.imwrite(save_path, segmented_image)

In [ ]:
df_test = df[:10]

# para testar o modelo de segmentação
for i, row in df_test.iterrows():
    file_name = row["file_name"]
    image_path = os.path.join(IMG_PATH, file_name)
    image = cv.cvtColor(cv.imread(image_path), cv.COLOR_BGR2RGB)
    
    segmented_image = cloth_segmentation(image, 0.5)
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(image)
    axes[0].set_title("Original Image")
    axes[0].axis("off")
    axes[1].imshow(segmented_image)
    axes[1].set_title("Segmented Image")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()